# Bonus Poin 1: Menggunakan Multi-Agent (AutoGen)

Bagian ini merupakan penyelesaian untuk **Bonus Poin 1** dari Mini Proyek 2, yaitu menggunakan platform multi-agent (seperti **AutoGen**) untuk melakukan penilaian terhadap jawaban essay siswa.

### Penjelasan Sistem Multi-Agent
Dalam sistem ini, kita akan membuat sebuah grup diskusi (*Group Chat*) yang terdiri dari beberapa agen AI untuk mengevaluasi setiap essay secara kolaboratif:
1. **Admin**: Bertugas memberikan essay kepada agen lain untuk dinilai.
2. **Penilai Utama**: Bertugas menganalisis tata bahasa dan isi essay, lalu memberikan perkiraan skor.
3. **Reviewer Senior**: Bertugas membaca analisis dari Penilai Utama dan memberikan keputusan mutlak berupa skor final (1-6).

Dengan pendekatan ini, penilaian diharapkan menjadi lebih objektif karena melalui proses diskusi dan review layaknya manusia.

### 1. Inisialisasi Agen (Admin, Penilai Utama, Reviewer Senior)
Pada tahap ini, kita mendefinisikan masing-masing agen dengan instruksi spesifik (*system message*) agar mereka tahu perannya. Kita menggunakan Ollama secara lokal sebagai backend LLM.

In [1]:
import autogen
import pandas as pd
from tqdm import tqdm
import re

# 1. Konfigurasi Ollama untuk AutoGen (Gunakan endpoint kompatibel OpenAI milik Ollama)
llm_config = {
    "config_list": [
        {
            "model": "llama3",
            "base_url": "http://localhost:11434/v1",
            "api_key": "ollama-local" # Wajib diisi meskipun dengan teks sembarang
        }
    ],
    "temperature": 0.1
}

# 2. Membuat Agen 1: Penilai Utama
penilai = autogen.AssistantAgent(
    name="Penilai_Utama",
    system_message="""Anda adalah Guru Penilai. Tugas Anda menganalisis esai yang diberikan dan memberikan perkiraan skor (1-6). 
    Berikan analisis singkat maksimal 3 kalimat mengenai tata bahasa dan isi esai, lalu akhiri dengan perkiraan skor.""",
    llm_config=llm_config,
)

# 3. Membuat Agen 2: Reviewer Senior (Pengambil Keputusan Final)
reviewer = autogen.AssistantAgent(
    name="Reviewer_Senior",
    system_message="""Anda adalah Reviewer Senior. Anda akan membaca analisis dari Penilai_Utama.
    Setuju atau tidak setuju dengan analisis tersebut, tugas Anda adalah menetapkan skor final.
    ATURAN MUTLAK: Anda HANYA boleh membalas dengan satu angka digit (1, 2, 3, 4, 5, atau 6). Jangan berikan teks lain.""",
    llm_config=llm_config,
)

# 4. Membuat Agen 3: Admin (Pembuat Tugas)
admin = autogen.UserProxyAgent(
    name="Admin",
    human_input_mode="NEVER", # Biarkan agen berjalan otomatis tanpa input manusia
    max_consecutive_auto_reply=1,
    code_execution_config=False,
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "")
)

print("Agen berhasil disiapkan!")

Agen berhasil disiapkan!


### 2. Fungsi Eksekusi Diskusi Agen
Fungsi `jalankan_diskusi_agen` ini akan menginisiasi percakapan antara Admin, Penilai Utama, dan Reviewer Senior. Percakapan dibatasi maksimal 3 putaran agar efisien, dan di akhir percakapan, kita mengekstrak skor final dari Reviewer Senior.

In [2]:
def jalankan_diskusi_agen(esai_teks):
    # Membersihkan riwayat percakapan sebelumnya agar agen tidak bingung
    admin.clear_history()
    penilai.clear_history()
    reviewer.clear_history()
    
    # Memulai Group Chat
    groupchat = autogen.GroupChat(
        agents=[admin, penilai, reviewer], 
        messages=[], 
        max_round=3 # Dibatasi 3 putaran: Admin bertanya -> Penilai menjawab -> Reviewer memutuskan
    )
    
    manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)
    
    # Admin memulai instruksi
    pesan_awal = f"Tolong nilai esai ini: \n\n{esai_teks}"
    
    # Menjalankan percakapan (ini akan mencetak log diskusi ke layar)
    admin.initiate_chat(manager, message=pesan_awal)
    
    # Mengambil pesan terakhir dari percakapan (seharusnya ini adalah angka dari Reviewer_Senior)
    pesan_terakhir = admin.chat_messages[manager][-1]['content']
    
    # Ekstraksi angka untuk memastikan output bersih
    match = re.search(r'[1-6]', pesan_terakhir)
    if match:
        return int(match.group())
    return 3 # Fallback

### 3. Uji Coba Multi-Agent pada Sebagian Data Latih
Sebelum menjalankan pada data *test*, mari kita simulasikan diskusi antar agen pada 5 sampel data latih untuk melihat bagaimana agen-agen tersebut berinteraksi dan memberikan skor.

In [3]:
# Ingat untuk menyesuaikan path karena dataset sekarang ada di folder terpisah
df_train = pd.read_csv('../dataset/train.csv').head(5) 

prediksi_multi_agent = []

print("Memulai proses Multi-Agent pada 5 esai...")

for index, row in tqdm(df_train.iterrows(), total=df_train.shape[0]):
    esai = row['full_text']
    
    # Jalankan diskusi antar agen
    skor_final = jalankan_diskusi_agen(esai)
    prediksi_multi_agent.append(skor_final)

df_train['pred_multi_agent'] = prediksi_multi_agent

print("\nSimulasi selesai! Berikut hasilnya:")
display(df_train[['essay_id', 'score', 'pred_multi_agent']])

Memulai proses Multi-Agent pada 5 esai...


  0%|          | 0/5 [00:00<?, ?it/s]

Admin (to chat_manager):

Tolong nilai esai ini: 

Many people have car where they live. The thing they don't know is that when you use a car alot of thing can happen like you can get in accidet or the smoke that the car has is bad to breath on if someone is walk but in VAUBAN,Germany they dont have that proble because 70 percent of vauban's families do not own cars,and 57 percent sold a car to move there. Street parkig ,driveways and home garages are forbidden on the outskirts of freiburd that near the French and Swiss borders. You probaly won't see a car in Vauban's streets because they are completely "car free" but If some that lives in VAUBAN that owns a car ownership is allowed,but there are only two places that you can park a large garages at the edge of the development,where a car owner buys a space but it not cheap to buy one they sell the space for you car for $40,000 along with a home. The vauban people completed this in 2006 ,they said that this an example of a growing trend

 20%|██        | 1/5 [00:22<01:30, 22.67s/it]

Admin (to chat_manager):

Tolong nilai esai ini: 

I am a scientist at NASA that is discussing the "face" on mars. I will be explaining how the "face" is a land form. By sharing my information about this isue i will tell you just that.

First off, how could it be a martions drawing. There is no plant life on mars as of rite now that we know of, which means so far as we know it is not possible for any type of life. That explains how it could not be made by martians. Also why and how would a martion build a face so big. It just does not make any since that a martian did this.

Next, why it is a landform. There are many landforms that are weird here in America, and there is also landforms all around the whole Earth. Many of them look like something we can relate to like a snake a turtle a human... So if there are landforms on earth dont you think landforms are on mars to? Of course! why not? It's just unique that the landform on Mars looks like a human face. Also if there was martians and

...

 80%|████████  | 4/5 [02:53<00:49, 49.99s/it]

Admin (to chat_manager):

Tolong nilai esai ini: 

Dear, State Senator

This is a letter to argue in favor of keeping the Electoral College."There are many reasons to keep the Electoral College" one reason is because it is widely regarded as an anachronism, a dispute over the outcome of an Electoral College vote is possible, but it is less likely than a dispute over the popular vote, and the Electoral College restores some of the weight in the political balance that large states (by population) lose by virue of the mal apportionment of the Senate decreed in the Constitution.

I am in favor of keeping the Electoral College because,it is widely regarded as an anachronism. A non-democratic method of selecting a president that ought to be [overruled] by declaring the canaditdate who receives the most populare votes the winner. The advocates of this position are correct in arguing that the Electoral College method is not democratic in a method sense.It is the electors who elect the the pres

100%|██████████| 5/5 [03:16<00:00, 39.36s/it]


Simulasi selesai! Berikut hasilnya:


,essay_id,score,pred_multi_agent
0,000d118,3,5
1,000fe60,3,5
2,001ab80,4,4
3,001bdc0,4,6
4,002ba53,3,5


### 4. Eksekusi pada Data Uji (Test Data) dan Format Submission
Setelah sistem multi-agent terbukti berjalan dengan baik, kita terapkan pada seluruh data uji (`test.csv`). Hasil akhirnya (berupa skor) akan digabungkan ke dalam format yang sesuai untuk submission Kaggle (`submission_autogen.csv`).

In [4]:
# 1. Muat data test
df_test = pd.read_csv('../dataset/test.csv')

prediksi_final_autogen = []

print("Memerintahkan Multi-Agent untuk menilai data uji Kaggle...")

# 2. Looping ke seluruh data test
for index, row in tqdm(df_test.iterrows(), total=df_test.shape[0]):
    esai_target = row['full_text']
    
    # Jalankan diskusi antar agen untuk setiap esai
    skor_final = jalankan_diskusi_agen(esai_target)
    prediksi_final_autogen.append(skor_final)

# 3. Format hasil sesuai aturan submission Kaggle
submission_autogen = pd.DataFrame({
    'essay_id': df_test['essay_id'],
    'score': prediksi_final_autogen
})

# 4. Simpan ke dalam folder outputs
submission_autogen.to_csv('../outputs/submission_autogen.csv', index=False)

print("\nSukses Besar! File 'submission_autogen.csv' sudah jadi dan siap di-submit ke Kaggle!")
display(submission_autogen.head())

Memerintahkan Multi-Agent untuk menilai data uji Kaggle...


  0%|          | 0/3 [00:00<?, ?it/s]

Admin (to chat_manager):

Tolong nilai esai ini: 

Many people have car where they live. The thing they don't know is that when you use a car alot of thing can happen like you can get in accidet or the smoke that the car has is bad to breath on if someone is walk but in VAUBAN,Germany they dont have that proble because 70 percent of vauban's families do not own cars,and 57 percent sold a car to move there. Street parkig ,driveways and home garages are forbidden on the outskirts of freiburd that near the French and Swiss borders. You probaly won't see a car in Vauban's streets because they are completely "car free" but If some that lives in VAUBAN that owns a car ownership is allowed,but there are only two places that you can park a large garages at the edge of the development,where a car owner buys a space but it not cheap to buy one they sell the space for you car for $40,000 along with a home. The vauban people completed this in 2006 ,they said that this an example of a growing trend

 33%|███▎      | 1/3 [00:25<00:51, 25.58s/it]

Admin (to chat_manager):

Tolong nilai esai ini: 

I am a scientist at NASA that is discussing the "face" on mars. I will be explaining how the "face" is a land form. By sharing my information about this isue i will tell you just that.

First off, how could it be a martions drawing. There is no plant life on mars as of rite now that we know of, which means so far as we know it is not possible for any type of life. That explains how it could not be made by martians. Also why and how would a martion build a face so big. It just does not make any since that a martian did this.

Next, why it is a landform. There are many landforms that are weird here in America, and there is also landforms all around the whole Earth. Many of them look like something we can relate to like a snake a turtle a human... So if there are landforms on earth dont you think landforms are on mars to? Of course! why not? It's just unique that the landform on Mars looks like a human face. Also if there was martians and

 67%|██████▋   | 2/3 [00:49<00:24, 24.70s/it]

Admin (to chat_manager):

Tolong nilai esai ini: 

People always wish they had the same technology that they have seen in movies, or the best new piece of technology that is all over social media. However, nobody seems to think of the risks that these kinds of new technologies may have. Cars have been around for many decades, and now manufacturers are starting to get on the bandwagon and come up with the new and improved technology that they hope will appeal to everyone. As of right now, it seems as though the negative characteristics of these cars consume the positive idea that these manufacturers have tried to convey.

Currently, this new technology in cars has a very long way to go before being completely "driverless". Drivers still need to be on alert when they are driving, as well as control the car near any accidents or complicated traffic situations. This seems to totally defeat the purpose of the "driverless" car. Eventually the technology may improve, but nobody can be certain

100%|██████████| 3/3 [01:58<00:00, 39.55s/it]


Sukses Besar! File 'submission_autogen.csv' sudah jadi dan siap di-submit ke Kaggle!


,essay_id,score
0,000d118,5
1,000fe60,5
2,001ab80,5
